# 03 Visualize Ground Truth

Purpose: create visual galleries of ground-truth bounding boxes before model training.

**Inputs**

- Prepared annotations at `FISHDETECT_PREPARED_ROOT/annotations_common.csv`
- Split manifest at `FISHDETECT_PREPARED_ROOT/metadata/split_manifest.csv`

**Outputs**

- `outputs/ground_truth_review/`

These galleries show VIAME/DIVE ground-truth boxes, not model predictions.


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Confirm Prepared Tables


In [ ]:
required = [
    PREPARED_ROOT / "annotations_common.csv",
    PREPARED_ROOT / "metadata" / "split_manifest.csv",
]
for path in required:
    print(f"{'OK' if path.exists() else 'MISSING':8} {path}")
if not all(path.exists() for path in required):
    raise FileNotFoundError("Prepared detection tables are missing. Run notebook 02 or scripts/prepare_dataset.py first.")


## Build Review Galleries

The script draws green boxes and class labels on source images. Modes cover random examples, rare classes, crowded images, and images containing multiple classes.


In [ ]:
!python scripts/visualize_ground_truth.py --config configs/experiments.yaml --mode random --n 50
!python scripts/visualize_ground_truth.py --config configs/experiments.yaml --mode rare --n 50
!python scripts/visualize_ground_truth.py --config configs/experiments.yaml --mode crowded --n 50
!python scripts/visualize_ground_truth.py --config configs/experiments.yaml --mode multi_class --n 50


## Inline Preview


In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown

review_root = Path("outputs/ground_truth_review")
preview_files = []
for mode in ["random", "rare", "crowded", "multi_class"]:
    preview_files.extend(sorted((review_root / mode).glob("*.jpg"))[:2])

if preview_files:
    for path in preview_files[:6]:
        display(Markdown(f"**{path.parent.name}: {path.name}**"))
        display(Image(filename=str(path), width=700))
else:
    print(f"No preview images found under {review_root}.")
